# 05 Error Analysis

This notebook performs validation-only error analysis across the implemented baseline and deep-learning model families, including the current Stage 14 fusion CNN, Stage 15 TCN ensemble, and Stage 16 temporal fusion TCN ensemble artifacts when available. The held-out test split is not loaded or evaluated here.

## Setup

The reusable error-analysis helpers accept validation prediction tables and produce comparable metrics, coverage summaries, error-type counts, participant summaries, transition-neighborhood diagnostics, confidence diagnostics, model-agreement summaries, and confusion-matrix figures.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.error_analysis import (
    DEFAULT_STAGE13_OUTPUT_DIR,
    build_stage6_validation_prediction_tables,
    discover_validation_prediction_files,
    load_discovered_predictions,
    materialize_deep_validation_predictions_from_checkpoints,
    run_stage13_error_analysis,
)

data_interim_dir = repo_root / "data/interim"
data_processed_dir = repo_root / "data/processed"
results_dir = repo_root / "results"
stage13_output_dir = repo_root / DEFAULT_STAGE13_OUTPUT_DIR
stage13_prediction_dir = stage13_output_dir / "predictions"

stage13_output_dir.mkdir(parents=True, exist_ok=True)
stage13_prediction_dir.mkdir(parents=True, exist_ok=True)
stage13_output_dir

## Optional Stage 6 Prediction Tables

The original Stage 6 notebook saves validation metrics and confusion matrices, but not epoch-level predictions. Enable this guarded cell to rebuild the existing Stage 6 model types from the saved train/validation feature tables and write prediction CSVs for validation error analysis. This uses only `features_train.csv` for fitting/CV and `features_val.csv` for prediction.

In [ ]:
BUILD_STAGE6_PREDICTIONS = False

if BUILD_STAGE6_PREDICTIONS:
    stage6_predictions = build_stage6_validation_prediction_tables(
        train_features_path=data_processed_dir / "features_train.csv",
        val_features_path=data_processed_dir / "features_val.csv",
        output_dir=stage13_prediction_dir,
        include_xgboost=True,
    )
    print(f"Saved {len(stage6_predictions)} Stage 6 prediction table(s).")
else:
    print("Skipping Stage 6 prediction rebuild. Set BUILD_STAGE6_PREDICTIONS = True when feature tables are available.")

## Optional Deep Prediction Export

Deep-learning runs from the current training utilities write `validation_epoch_predictions.csv`, `validation_aggregated_epoch_predictions_*.csv`, or ensemble validation prediction files depending on model family. For routine error analysis, leave the export flag below as `False` and first inspect the discovery table. Only enable it if a completed validation run has a saved checkpoint but is missing its validation prediction CSV; the helper performs validation inference only and records per-run errors in the summary table instead of using the test split.

In [ ]:
MATERIALIZE_DEEP_PREDICTIONS_FROM_CHECKPOINTS = False

if MATERIALIZE_DEEP_PREDICTIONS_FROM_CHECKPOINTS:
    print("Exporting missing validation predictions from saved checkpoints; no training or test evaluation is run.")
    materialization_summary = materialize_deep_validation_predictions_from_checkpoints(
        results_dir=results_dir,
        overwrite=False,
    )
    display(materialization_summary)
    if not materialization_summary.empty and "status" in materialization_summary:
        failed_exports = materialization_summary[materialization_summary["status"] == "error"]
        if not failed_exports.empty:
            print("One or more checkpoint exports failed; review the error rows before continuing.")
else:
    print("Skipping checkpoint-based deep prediction export. First inspect prediction_discovery below; set MATERIALIZE_DEEP_PREDICTIONS_FROM_CHECKPOINTS = True only for completed runs that have checkpoints but no validation prediction CSV.")

## Discover Validation Predictions

This discovery step includes Stage 6 prediction tables, single-output deep prediction tables, Stage 12 aggregated many-to-many prediction tables, and current Stage 14/15/16 validation prediction artifacts when available.

In [ ]:
prediction_discovery = discover_validation_prediction_files(results_dir=results_dir)
prediction_discovery

## Load Epoch Metadata

`epoch_index.csv` adds participant identity, valid validation coverage, missingness columns, and true temporal transition context. If it is unavailable, the notebook still runs the model-level analyses that only require prediction tables.

In [ ]:
epoch_index_path = data_interim_dir / "epoch_index.csv"
if epoch_index_path.exists():
    epoch_index = pd.read_csv(epoch_index_path, dtype={"participant_id": str})
    print(f"Loaded epoch index with {len(epoch_index):,} row(s).")
else:
    epoch_index = pd.DataFrame()
    print(f"Epoch index not found at {epoch_index_path}; metadata-linked summaries will be skipped or empty.")

## Run Validation Error Analysis

This cell writes validation error-analysis artifacts under `results/stage13_error_analysis/`. It is validation-only: it analyzes whatever validation prediction tables are available and does not load the held-out test feature table or test epochs.

In [ ]:
if prediction_discovery.empty:
    print("No validation error-analysis-compatible prediction files were found yet.")
    print("Run completed model stages after this implementation, or enable the Stage 6 prediction rebuild above.")
    stage13_outputs = {}
else:
    validation_predictions = load_discovered_predictions(prediction_discovery)
    stage13_outputs = run_stage13_error_analysis(
        validation_predictions,
        epoch_index=epoch_index,
        output_dir=stage13_output_dir,
        transition_radius=2,
        high_confidence_threshold=0.80,
        make_plots=True,
    )
    print(f"Analyzed {len(validation_predictions):,} validation prediction row(s).")
    print(f"Saved validation error-analysis artifacts to {stage13_output_dir.resolve()}.")
    display(stage13_outputs["model_validation_metrics"])
    display(stage13_outputs["model_coverage"])

## Error Patterns

Review the most common true-label to predicted-label mistakes, participant-level failures, temporal transition effects, and high-confidence errors. These outputs are intended to guide backup modeling choices such as richer features, calibration, temporal smoothing, or simple ensembling.

In [ ]:
if stage13_outputs:
    display(stage13_outputs["error_type_summary"].head(20))
    display(stage13_outputs["participant_error_summary"].head(20))
    if "temporal_error_summary" in stage13_outputs:
        display(stage13_outputs["temporal_error_summary"])
    display(stage13_outputs["confidence_summary"])
    display(stage13_outputs["model_disagreement_summary"])
    display(stage13_outputs["shared_epoch_model_metrics"])
else:
    print("Run the validation error-analysis cell above after prediction tables are available.")

## Interpretation Checklist

Use the saved tables and figures to answer: which classes dominate errors; whether errors are concentrated in a few participants; whether context/sequence models reduce transition-adjacent errors; whether tabular and neural models make complementary mistakes; and whether the next backup should prioritize enhanced features, temporal smoothing, calibration, ensembling, or preprocessing changes.